**Kagglehub Download**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'us-accidents' dataset.
Path to dataset files: /kaggle/input/us-accidents


**Check the exact filename**

In [ ]:
import pandas as pd
import os

print(os.listdir(path))

['US_Accidents_March23.csv']


**Down to 40 columns**

In [ ]:
import pandas as pd
import os

cols_needed = [
    # Target
    'Severity',
    # Time
    'Start_Time', 'End_Time',
    # Location
    'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)',
    'Street', 'City', 'County', 'State', 'Zipcode', 'Timezone', 'Airport_Code',
    # Weather
    'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)',
    'Precipitation(in)', 'Weather_Condition',
    # Road features (POI flags)
    'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit',
    'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming',
    'Traffic_Signal', 'Turning_Loop',
    # Light conditions
    'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight'
]

file_path = os.path.join(path, "US_Accidents_March23.csv")

chunks = []
for chunk in pd.read_csv(file_path, usecols=cols_needed, chunksize=200_000, low_memory=False):
    chunk['Start_Time'] = pd.to_datetime(chunk['Start_Time'], format='mixed', errors='coerce')
    chunk['End_Time'] = pd.to_datetime(chunk['End_Time'], format='mixed', errors='coerce')
    # 2022-2023 only — cuts rows roughly in half vs 2020-2023
    filtered = chunk[(chunk['Start_Time'].dt.year >= 2022) & (chunk['Start_Time'].dt.year <= 2023)]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)

(2009085, 41)


**Sample Down**

In [ ]:
from sklearn.model_selection import train_test_split

print(df['Severity'].value_counts(normalize=True))  # look before you cut

df_sample, _ = train_test_split(
    df, train_size=300_000, stratify=df['Severity'], random_state=42
)
df_sample = df_sample.reset_index(drop=True)
print(df_sample.shape)
print(df_sample['Severity'].value_counts())

Severity
2    0.917292
3    0.038802
4    0.025054
1    0.018852
Name: proportion, dtype: float64
(300000, 41)
Severity
2    275188
3     11641
4      7516
1      5655
Name: count, dtype: int64


**Check Missingness**

In [ ]:
# Missingness as a percentage — easier to eyeball than raw counts
miss = (df_sample.isnull().sum() / len(df_sample) * 100).sort_values(ascending=False)
print(miss[miss > 0])

End_Lng                  11.697667
End_Lat                  11.697667
Precipitation(in)         3.803667
Wind_Chill(F)             3.160333
Wind_Direction            2.827333
Wind_Speed(mph)           2.827000
Visibility(mi)            2.439667
Humidity(%)               2.399667
Weather_Condition         2.271667
Temperature(F)            2.262333
Pressure(in)              1.969667
Civil_Twilight            0.841000
Nautical_Twilight         0.841000
Astronomical_Twilight     0.841000
Sunrise_Sunset            0.841000
Street                    0.422000
Airport_Code              0.406000
Timezone                  0.099000
Zipcode                   0.022000
City                      0.002667
dtype: float64


**Drop rows with any remaining missing values**

In [ ]:
# Drop the two weakest, least necessary columns
df_sample = df_sample.drop(columns=['End_Lat', 'End_Lng'])

# Drop rows with any remaining missing values
before = len(df_sample)
df_sample = df_sample.dropna()
after = len(df_sample)
print(f"Dropped {before - after} rows ({(before-after)/before*100:.2f}%)")

# Confirm class balance survived the drop
print(df_sample['Severity'].value_counts(normalize=True))

Dropped 20657 rows (6.89%)
Severity
2    0.918502
3    0.038673
4    0.023709
1    0.019116
Name: proportion, dtype: float64


**Temporal features**

In [ ]:
df_sample['Year'] = df_sample['Start_Time'].dt.year
df_sample['Month'] = df_sample['Start_Time'].dt.month
df_sample['Hour'] = df_sample['Start_Time'].dt.hour
df_sample['DayOfWeek'] = df_sample['Start_Time'].dt.dayofweek
df_sample['Is_Weekend'] = df_sample['DayOfWeek'].isin([5, 6]).astype(int)
df_sample['Duration_Minutes'] = (df_sample['End_Time'] - df_sample['Start_Time']).dt.total_seconds() / 60

def get_season(m):
    if m in [12, 1, 2]: return 'Winter'
    if m in [3, 4, 5]: return 'Spring'
    if m in [6, 7, 8]: return 'Summer'
    return 'Fall'
df_sample['Season'] = df_sample['Month'].apply(get_season)

df_sample['Is_Rush_Hour'] = (
    (~df_sample['Is_Weekend'].astype(bool)) &
    (df_sample['Hour'].isin([7, 8, 9, 16, 17, 18]))
).astype(int)

print(df_sample['Duration_Minutes'].describe())

count    279343.000000
mean        339.520710
std       10424.289302
min           2.533333
25%          63.000000
50%          81.450000
75%         129.483333
max      527173.000000
Name: Duration_Minutes, dtype: float64


**Cap at a realistic ceiling.**

In [ ]:
# Cap Duration_Minutes at 24 hours (1440 min) — anything beyond is treated as bad data
before = len(df_sample)
df_sample = df_sample[df_sample['Duration_Minutes'] <= 1440]
after = len(df_sample)
print(f"Removed {before - after} rows ({(before-after)/before*100:.2f}%) with implausible duration")

print(df_sample['Duration_Minutes'].describe())
print(df_sample['Severity'].value_counts(normalize=True))

Removed 1837 rows (0.66%) with implausible duration
count    277506.000000
mean        108.869688
std          99.905788
min           2.533333
25%          62.116667
50%          81.166667
75%         129.050000
max        1440.000000
Name: Duration_Minutes, dtype: float64
Severity
2    0.918629
3    0.038918
4    0.023210
1    0.019243
Name: proportion, dtype: float64


**Handling the categorical columns before encoding**

A few of the columns are high-cardinality identifiers rather than useful predictive features — Street (near-unique per row), Zipcode (thousands of values), Airport_Code, and County add noise and blow up dimensionality without helping the model generalize.

In [ ]:
df_sample = df_sample.drop(columns=['Street', 'Zipcode', 'Airport_Code', 'County'], errors='ignore')
print(df_sample.shape)

(277506, 43)


**Converting the boolean POI flags to actual 0/1 integers**
(they're currently True/False strings/objects from the CSV)

In [ ]:
bool_cols = ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit',
             'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming',
             'Traffic_Signal', 'Turning_Loop']
df_sample[bool_cols] = df_sample[bool_cols].astype(int)

Label-encode the remaining true categoricals (same approach Tang et al. research paper used)

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['State', 'Timezone', 'Wind_Direction', 'Weather_Condition',
            'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
            'Astronomical_Twilight', 'Season', 'City']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_sample[col] = le.fit_transform(df_sample[col].astype(str))
    encoders[col] = le  # need them later to decode predictions in your web app

**Standardize the numeric columns**

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ['Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)',
            'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)',
            'Precipitation(in)', 'Duration_Minutes']

scaler = StandardScaler()
df_sample[num_cols] = scaler.fit_transform(df_sample[num_cols])

print(df_sample.dtypes)
print(df_sample.shape)

Severity                          int64
Start_Time               datetime64[ns]
End_Time                 datetime64[ns]
Start_Lat                       float64
Start_Lng                       float64
Distance(mi)                    float64
City                              int64
State                             int64
Timezone                          int64
Temperature(F)                  float64
Wind_Chill(F)                   float64
Humidity(%)                     float64
Pressure(in)                    float64
Visibility(mi)                  float64
Wind_Direction                    int64
Wind_Speed(mph)                 float64
Precipitation(in)               float64
Weather_Condition                 int64
Amenity                           int64
Bump                              int64
Crossing                          int64
Give_Way                          int64
Junction                          int64
No_Exit                           int64
Railway                           int64


Drop Start_Time/End_Time [already extracted everything useful from them (Year, Month, Hour, DayOfWeek, Duration_Minutes), and raw datetime objects can't go into sklearn models directly]

In [ ]:
df_final = df_sample.drop(columns=['Start_Time', 'End_Time'])
print(df_final.shape)  # should be (277506, 41)

(277506, 41)


**80/20 Split**

In [ ]:
from sklearn.model_selection import train_test_split

X = df_final.drop(columns=['Severity'])
y = df_final['Severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Train: (222004, 40) Test: (55502, 40)
Severity
2    0.918628
3    0.038918
4    0.023211
1    0.019243
Name: proportion, dtype: float64
Severity
2    0.918634
3    0.038918
4    0.023206
1    0.019243
Name: proportion, dtype: float64


**Training both models**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# XGBoost needs labels starting at 0, not 1
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                     random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train_xgb)

print("Both models trained")

Both models trained


**Precision, Accuracy, Recall, F1, AUC-ROC, Confusion Matrix**

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)
import numpy as np

# --- Random Forest predictions ---
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)

# --- XGBoost predictions (remember: labels are shifted -1, shift back for reporting) ---
xgb_pred_raw = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)
xgb_pred = xgb_pred_raw + 1  # shift back to original Severity 1-4

def evaluate_model(name, y_true, y_pred, y_proba):
    print(f"\n===== {name} =====")
    print("Accuracy: ", accuracy_score(y_true, y_pred))
    print("Precision (macro):", precision_score(y_true, y_pred, average='macro', zero_division=0))
    print("Recall (macro):   ", recall_score(y_true, y_pred, average='macro', zero_division=0))
    print("F1 (macro):       ", f1_score(y_true, y_pred, average='macro', zero_division=0))
    print("Precision (weighted):", precision_score(y_true, y_pred, average='weighted', zero_division=0))
    print("Recall (weighted):   ", recall_score(y_true, y_pred, average='weighted', zero_division=0))
    print("F1 (weighted):       ", f1_score(y_true, y_pred, average='weighted', zero_division=0))
    print("AUC-ROC (macro, OVR):", roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro'))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nFull classification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

evaluate_model("Random Forest", y_test, rf_pred, rf_proba)
evaluate_model("XGBoost", y_test, xgb_pred, xgb_proba)


===== Random Forest =====
Accuracy:  0.9390112068033585
Precision (macro): 0.6667222127862136
Recall (macro):    0.5641634200714563
F1 (macro):        0.5879959579962131
Precision (weighted): 0.9246155188688443
Recall (weighted):    0.9390112068033585
F1 (weighted):        0.9288125789699401
AUC-ROC (macro, OVR): 0.9384079756071131

Confusion Matrix:
[[  753   250    65     0]
 [  243 50191   396   156]
 [  108   956  1096     0]
 [    1  1203     7    77]]

Full classification report:
              precision    recall  f1-score   support

           1       0.68      0.71      0.69      1068
           2       0.95      0.98      0.97     50986
           3       0.70      0.51      0.59      2160
           4       0.33      0.06      0.10      1288

    accuracy                           0.94     55502
   macro avg       0.67      0.56      0.59     55502
weighted avg       0.92      0.94      0.93     55502


===== XGBoost =====
Accuracy:  0.9400021620842492
Precision (macro): 0.7

**Testing both with class weighting**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report)
import numpy as np

# --- Random Forest with balanced class weights ---
rf_bal = RandomForestClassifier(n_estimators=100, max_depth=None,
                                 class_weight='balanced', random_state=42, n_jobs=-1)
rf_bal.fit(X_train, y_train)
print("RF (balanced) trained")

# --- XGBoost with sample weights (XGBoost has no built-in class_weight, so compute manually) ---
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_xgb)

xgb_bal = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                         random_state=42, eval_metric='mlogloss')
xgb_bal.fit(X_train, y_train_xgb, sample_weight=sample_weights)
print("XGBoost (balanced) trained")

RF (balanced) trained
XGBoost (balanced) trained


In [ ]:
def evaluate_model(name, y_true, y_pred, y_proba):
    print(f"\n===== {name} =====")
    print("Accuracy: ", accuracy_score(y_true, y_pred))
    print("Precision (macro):", precision_score(y_true, y_pred, average='macro', zero_division=0))
    print("Recall (macro):   ", recall_score(y_true, y_pred, average='macro', zero_division=0))
    print("F1 (macro):       ", f1_score(y_true, y_pred, average='macro', zero_division=0))
    print("F1 (weighted):    ", f1_score(y_true, y_pred, average='weighted', zero_division=0))
    print("AUC-ROC (macro, OVR):", roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro'))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nFull classification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

# RF balanced
rf_bal_pred = rf_bal.predict(X_test)
rf_bal_proba = rf_bal.predict_proba(X_test)
evaluate_model("Random Forest (balanced)", y_test, rf_bal_pred, rf_bal_proba)

# XGBoost balanced
xgb_bal_pred = xgb_bal.predict(X_test) + 1
xgb_bal_proba = xgb_bal.predict_proba(X_test)
evaluate_model("XGBoost (balanced)", y_test, xgb_bal_pred, xgb_bal_proba)


===== Random Forest (balanced) =====
Accuracy:  0.9361824799106339
Precision (macro): 0.6561061315683715
Recall (macro):    0.5361806207102995
F1 (macro):        0.5660195462076217
F1 (weighted):     0.924676700077966
AUC-ROC (macro, OVR): 0.9360228216325202

Confusion Matrix:
[[  715   294    59     0]
 [  239 50234   360   153]
 [  111  1108   941     0]
 [    2  1212     4    70]]

Full classification report:
              precision    recall  f1-score   support

           1       0.67      0.67      0.67      1068
           2       0.95      0.99      0.97     50986
           3       0.69      0.44      0.53      2160
           4       0.31      0.05      0.09      1288

    accuracy                           0.94     55502
   macro avg       0.66      0.54      0.57     55502
weighted avg       0.92      0.94      0.92     55502


===== XGBoost (balanced) =====
Accuracy:  0.7424417138121149
Precision (macro): 0.4845895037135169
Recall (macro):    0.8191310856661266
F1 (macro)

Save the plain (unweighted) Random Forest + everything needed for the web app

In [ ]:
import pickle

# Final model — plain Random Forest
with open('best_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

# Preprocessing objects — the web app needs these to transform new user inputs
# the exact same way the training data was transformed
with open('encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Also save the exact column order/names — critical so your web app
# builds input rows in the same order the model expects
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)

print("Saved: best_model.pkl, encoders.pkl, scaler.pkl, feature_columns.pkl")

Saved: best_model.pkl, encoders.pkl, scaler.pkl, feature_columns.pkl


In [ ]:
import pickle
import pandas as pd

# Load everything back in (proves the .pkl files actually work standalone)
with open('best_model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('feature_columns.pkl', 'rb') as f:
    feature_columns = pickle.load(f)

print("All files loaded successfully")

All files loaded successfully


In [ ]:
import os
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

os.makedirs("output", exist_ok=True)

def summarize(name, y_true, y_pred, y_proba):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "auc_roc_ovr": roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro"),
    }

rows = [
    summarize("Random Forest", y_test, rf_pred, rf_proba),
    summarize("XGBoost", y_test, xgb_pred, xgb_proba),
    summarize("Random Forest (balanced)", y_test, rf_bal_pred, rf_bal_proba),
    summarize("XGBoost (balanced)", y_test, xgb_bal_pred, xgb_bal_proba),
]

df_results = pd.DataFrame(rows).round(4)
df_results.to_csv("output/model_results.csv", index=False)
print(df_results)

                      model  accuracy  precision_macro  recall_macro  \
0             Random Forest    0.9390           0.6667        0.5642   
1                   XGBoost    0.9400           0.7082        0.5761   
2  Random Forest (balanced)    0.9362           0.6561        0.5362   
3        XGBoost (balanced)    0.7424           0.4846        0.8191   

   f1_macro  f1_weighted  auc_roc_ovr  
0    0.5880       0.9288       0.9384  
1    0.5759       0.9289       0.9498  
2    0.5660       0.9247       0.9360  
3    0.5389       0.8114       0.9460  


In [ ]:
from google.colab import files
files.download("output/model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
sample_batch = X_test.sample(n=10, random_state=None)  # random_state=None = different each run
sample_indices = sample_batch.index

predictions = model.predict(sample_batch)
actual = y_test.loc[sample_indices]

results = pd.DataFrame({
    'Predicted_Severity': predictions,
    'Actual_Severity': actual.values
})
print(results)

   Predicted_Severity  Actual_Severity
0                   2                2
1                   2                2
2                   2                2
3                   3                3
4                   3                3
5                   3                3
6                   2                2
7                   2                2
8                   2                2
9                   2                2


In [ ]:
demo_rows = pd.concat([
    X_test[y_test == s].sample(1, random_state=1) for s in [1, 2, 3, 4]
])
demo_actual = y_test.loc[demo_rows.index]
demo_pred = model.predict(demo_rows)

print(pd.DataFrame({'Predicted': demo_pred, 'Actual': demo_actual.values}, index=demo_rows.index))

        Predicted  Actual
227678          1       1
214724          2       2
87517           3       3
86624           2       4


In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

temporal_cols = ['Year', 'Month', 'Hour', 'DayOfWeek', 'Is_Weekend', 'Season', 'Is_Rush_Hour']

def run_experiment(X_tr, X_te, y_tr, y_te, label, use_smote=True):
    if use_smote:
        smote = SMOTE(random_state=42)
        X_tr, y_tr = smote.fit_resample(X_tr, y_tr)

    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)

    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)

    print(f"\n===== {label} =====")
    print("Accuracy:     ", accuracy_score(y_te, pred))
    print("Macro Prec:   ", precision_score(y_te, pred, average='macro', zero_division=0))
    print("Macro Recall: ", recall_score(y_te, pred, average='macro', zero_division=0))
    print("Macro F1:     ", f1_score(y_te, pred, average='macro', zero_division=0))
    print("Weighted F1:  ", f1_score(y_te, pred, average='weighted', zero_division=0))
    print("AUC-ROC:      ", roc_auc_score(y_te, proba, multi_class='ovr', average='macro'))
    print("Per-class recall:", recall_score(y_te, pred, average=None))
    return model

# 1. Baseline — no SMOTE, no temporal features (closest to what the paper likely had)
X_train_no_t = X_train.drop(columns=temporal_cols)
X_test_no_t = X_test.drop(columns=temporal_cols)
run_experiment(X_train_no_t, X_test_no_t, y_train, y_test, "No temporal, no SMOTE", use_smote=False)

# 2. SMOTE only — matches the paper's imbalance-handling approach
run_experiment(X_train_no_t, X_test_no_t, y_train, y_test, "No temporal, WITH SMOTE", use_smote=True)

# 3. Temporal features only — your contribution, no SMOTE
run_experiment(X_train, X_test, y_train, y_test, "WITH temporal, no SMOTE", use_smote=False)

# 4. Both combined — your contribution + matching the paper's technique
run_experiment(X_train, X_test, y_train, y_test, "WITH temporal, WITH SMOTE", use_smote=True)


===== No temporal, no SMOTE =====
Accuracy:      0.9347771251486433
Macro Prec:    0.6443761818283843
Macro Recall:  0.5155151237115408
Macro F1:      0.5532228204648869
Weighted F1:   0.9237229544060968
AUC-ROC:       0.935717057692824
Per-class recall: [0.52715356 0.98411329 0.49722222 0.05357143]

===== No temporal, WITH SMOTE =====
Accuracy:      0.9046701019783071
Macro Prec:    0.5365339021874744
Macro Recall:  0.663470613901842
Macro F1:      0.5861535337554328
Weighted F1:   0.9122825697952579
AUC-ROC:       0.9280103993313744
Per-class recall: [0.74344569 0.93147138 0.76157407 0.2173913 ]

===== WITH temporal, no SMOTE =====
Accuracy:      0.9390112068033585
Macro Prec:    0.6667222127862136
Macro Recall:  0.5641634200714563
Macro F1:      0.5879959579962131
Weighted F1:   0.9288125789699401
AUC-ROC:       0.9384079947833177
Per-class recall: [0.70505618 0.98440748 0.50740741 0.05978261]

===== WITH temporal, WITH SMOTE =====
Accuracy:      0.9158048358617707
Macro Prec:    0

RandomForestClassifier(n_jobs=-1, random_state=42)

**Training Decision Tree and MLP (Neural Network)**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)

def evaluate_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)
    print(f"\n===== {name} =====")
    print("Accuracy:     ", accuracy_score(y_te, pred))
    print("Macro Prec:   ", precision_score(y_te, pred, average='macro', zero_division=0))
    print("Macro Recall: ", recall_score(y_te, pred, average='macro', zero_division=0))
    print("Macro F1:     ", f1_score(y_te, pred, average='macro', zero_division=0))
    print("Weighted F1:  ", f1_score(y_te, pred, average='weighted', zero_division=0))
    print("AUC-ROC:      ", roc_auc_score(y_te, proba, multi_class='ovr', average='macro'))
    print("Per-class recall:", recall_score(y_te, pred, average=None))
    print(confusion_matrix(y_te, pred))

# --- Decision Tree ---
dt = DecisionTreeClassifier(max_depth=15, random_state=42)
dt.fit(X_train, y_train)
evaluate_model("Decision Tree", dt, X_test, y_test)

# --- MLP (Neural Network) ---
# MLP is sensitive to scale, but you've already standardized numeric columns, so it's ready
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=100, random_state=42, early_stopping=True)
mlp.fit(X_train, y_train)
evaluate_model("MLP Neural Network", mlp, X_test, y_test)


===== Decision Tree =====
Accuracy:      0.9304529566502108
Macro Prec:    0.6055059519087581
Macro Recall:  0.5711862764239228
Macro F1:      0.5811287464267985
Weighted F1:   0.9243903549160872
AUC-ROC:       0.8529988298058188
Per-class recall: [0.65917603 0.97299259 0.56018519 0.0923913 ]
[[  704   248   116     0]
 [  260 49609   717   400]
 [  118   827  1210     5]
 [    3  1155    11   119]]

===== MLP Neural Network =====
Accuracy:      0.9186335627544954
Macro Prec:    0.22965839068862384
Macro Recall:  0.25
Macro F1:      0.23939786642626398
Weighted F1:   0.8796756598039347
AUC-ROC:       0.5617138599341205
Per-class recall: [0. 1. 0. 0.]
[[    0  1068     0     0]
 [    0 50986     0     0]
 [    0  2160     0     0]
 [    0  1288     0     0]]


**Improving MLP Results**

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.neural_network import MLPClassifier

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

mlp_sm = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    early_stopping=True,
    n_iter_no_change=15,
    random_state=42
)
mlp_sm.fit(X_train_sm, y_train_sm)

evaluate_model("MLP Neural Network (SMOTE)", mlp_sm, X_test, y_test)


===== MLP Neural Network (SMOTE) =====
Accuracy:      0.7370725379265612
Macro Prec:    0.41006175614246704
Macro Recall:  0.6751268623423629
Macro F1:      0.4529799441419349
Weighted F1:   0.802549620455978
AUC-ROC:       0.8777530175184091
Per-class recall: [0.85018727 0.74596948 0.62453704 0.47981366]
[[  908    98    49    13]
 [ 1804 38034  2971  8177]
 [  383   381  1349    47]
 [   17   627    26   618]]


# Part 2: Deployable model (leak-free preprocessing, no post-accident features)

Why this part exists: the web app can only use inputs a user knows *before* an accident is analysed. `Duration_Minutes` and `Distance(mi)` are only known afterwards, and `City` has thousands of labels. This part:
1. rebuilds the same 277,506-row dataset and the same 80/20 split as Part 1,
2. fits encoders and scaler **inside a pipeline, on the training split only** (fixes the leakage noted in Part 1),
3. drops `Duration_Minutes`, `Distance(mi)`, `City` and `Year`,
4. re-runs the temporal x SMOTE ablation and the 4-model comparison on that feature set,
5. exports one `model_bundle.joblib`, a results CSV, a confusion matrix, and a generated Angular `feature-schema.ts`.

Run the cells from the top of the notebook first (they create `df`).

In [ ]:
# Rebuild the dataset WITHOUT fitting any encoder or scaler yet.
# Needs `df` from the "Down to 40 columns" cell.
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SAMPLE_N = 300_000
d, _ = train_test_split(df, train_size=SAMPLE_N, stratify=df['Severity'], random_state=42)
d = d.reset_index(drop=True)
d = d.drop(columns=['End_Lat', 'End_Lng']).dropna()                # same rows as Part 1
d['Duration_Minutes'] = (d['End_Time'] - d['Start_Time']).dt.total_seconds() / 60
d = d[d['Duration_Minutes'] <= 1440]                                # same bad-data filter as Part 1
print("Rows:", len(d), "(Part 1 had 277,506)")

# Temporal features. Year is deliberately left out: a model that keys on 2022/2023
# cannot generalise to dates the user enters in the app.
def season_of(m):
    if m in (12, 1, 2): return 'Winter'
    if m in (3, 4, 5):  return 'Spring'
    if m in (6, 7, 8):  return 'Summer'
    return 'Fall'

d['Month'] = d['Start_Time'].dt.month
d['Hour'] = d['Start_Time'].dt.hour
d['DayOfWeek'] = d['Start_Time'].dt.dayofweek
d['Is_Weekend'] = d['DayOfWeek'].isin([5, 6]).astype(int)
d['Season'] = d['Month'].map(season_of)
d['Is_Rush_Hour'] = ((d['Is_Weekend'] == 0) & d['Hour'].isin([7, 8, 9, 16, 17, 18])).astype(int)

bool_cols = ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit',
             'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming',
             'Traffic_Signal', 'Turning_Loop']
d[bool_cols] = d[bool_cols].astype(int)
d = d.drop(columns=['Start_Time', 'End_Time', 'Street', 'Zipcode', 'Airport_Code', 'County'])

X_all = d.drop(columns=['Severity'])
y_all = d['Severity']
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)
print("Train:", Xd_train.shape, "Test:", Xd_test.shape)
print(X_all.columns.tolist())

In [ ]:
# Pipeline builder, model definitions and a single-run helper.
import gc, time
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

POST_ACCIDENT = ['Duration_Minutes', 'Distance(mi)', 'City']        # not available to a real user
TEMPORAL = ['Month', 'Hour', 'DayOfWeek', 'Is_Weekend', 'Season', 'Is_Rush_Hour']
BINARY = bool_cols + ['Is_Weekend', 'Is_Rush_Hour']
BASE_COLS = [c for c in X_all.columns if c not in POST_ACCIDENT + TEMPORAL]
print("Deployable features:", len(BASE_COLS) + len(TEMPORAL),
      "(base", len(BASE_COLS), "+ temporal", len(TEMPORAL), ")")

def make_pre(cols):
    cat = [c for c in cols if not pd.api.types.is_numeric_dtype(X_all[c])]
    binary = [c for c in cols if c in BINARY]
    num = [c for c in cols if c not in cat + binary]
    return ColumnTransformer([
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat),
        ('num', StandardScaler(), num),
        ('bin', 'passthrough', binary),
    ])

# min_samples_leaf keeps the saved forest small enough for GitHub (100 MB limit).
# If model_bundle.joblib is still too big, raise it (5 -> 10 -> 20) and re-run.
RF_PARAMS = dict(n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1)
MODELS = {
    'RF':  lambda: RandomForestClassifier(**RF_PARAMS),
    'XGB': lambda: XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                 random_state=42, eval_metric='mlogloss', n_jobs=-1),
    'DT':  lambda: DecisionTreeClassifier(max_depth=15, random_state=42),
    'MLP': lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300,
                                 early_stopping=True, n_iter_no_change=15, random_state=42),
}

def run_one(label, model_key, temporal=True, smote=True, extra=()):
    cols = BASE_COLS + (TEMPORAL if temporal else []) + list(extra)
    pre = make_pre(cols)
    Xtr = pre.fit_transform(Xd_train[cols])          # fit on TRAIN only
    Xte = pre.transform(Xd_test[cols])
    ytr = yd_train
    if smote:                                        # resample the training data only
        Xtr, ytr = SMOTE(random_state=42).fit_resample(Xtr, ytr)
    model = MODELS[model_key]()
    off = 1 if model_key == 'XGB' else 0             # XGBoost needs labels 0..3
    t0 = time.time()
    model.fit(Xtr, ytr - off)
    secs = time.time() - t0
    pred = model.predict(Xte) + off
    proba = model.predict_proba(Xte)
    rec = recall_score(yd_test, pred, average=None, labels=[1, 2, 3, 4], zero_division=0)
    row = {
        'run': label, 'n_features': len(cols),
        'accuracy': accuracy_score(yd_test, pred),
        'precision_macro': precision_score(yd_test, pred, average='macro', zero_division=0),
        'recall_macro': recall_score(yd_test, pred, average='macro', zero_division=0),
        'f1_macro': f1_score(yd_test, pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(yd_test, pred, average='weighted', zero_division=0),
        'auc_roc_ovr': roc_auc_score(yd_test, proba, multi_class='ovr', average='macro'),
        'recall_sev1': rec[0], 'recall_sev2': rec[1], 'recall_sev3': rec[2], 'recall_sev4': rec[3],
        'fit_seconds': secs,
    }
    pipe = Pipeline([('preprocess', pre), ('model', model)])
    return row, pipe, cols, pred

In [ ]:
# All runs on the deployable feature set. This is slow (SMOTE grows the training set to ~800k rows).
# Trim RUNS if you are short on time, but keep the four RF rows: they are the ablation.
RUNS = [
    # (label, model, temporal features?, SMOTE?)
    ('RF | no temporal | no SMOTE',  'RF',  False, False),
    ('RF | no temporal | SMOTE',     'RF',  False, True),
    ('RF | temporal | no SMOTE',     'RF',  True,  False),
    ('RF | temporal | SMOTE',        'RF',  True,  True),
    ('XGB | temporal | no SMOTE',    'XGB', True,  False),
    ('XGB | temporal | SMOTE',       'XGB', True,  True),
    ('DT | temporal | no SMOTE',     'DT',  True,  False),
    ('DT | temporal | SMOTE',        'DT',  True,  True),
    ('MLP | temporal | no SMOTE',    'MLP', True,  False),
    ('MLP | temporal | SMOTE',       'MLP', True,  True),
]
KEEP = 'RF | temporal | SMOTE'     # the configuration that gets deployed

results, fitted = [], {}
for label, key, temporal, smote in RUNS:
    row, pipe, cols, pred = run_one(label, key, temporal, smote)
    results.append(row)
    print(f"{label:30s} acc={row['accuracy']:.3f}  macroF1={row['f1_macro']:.3f}  "
          f"macroRec={row['recall_macro']:.3f}  AUC={row['auc_roc_ovr']:.3f}  "
          f"recall(1/2/3/4)={row['recall_sev1']:.2f}/{row['recall_sev2']:.2f}/"
          f"{row['recall_sev3']:.2f}/{row['recall_sev4']:.2f}  [{row['fit_seconds']:.0f}s]")
    if label == KEEP:
        fitted[label] = {'pipe': pipe, 'cols': cols, 'pred': pred}
    del pipe
    gc.collect()

import os
os.makedirs('output', exist_ok=True)
df_res = pd.DataFrame(results).round(4)
df_res.to_csv('output/model_results_deployable.csv', index=False)
df_res

In [ ]:
# Optional comparison row: same best configuration WITH the post-accident features (Duration, Distance, City).
# Shows how much of the earlier score came from information a real user would not have.
row, _, _, _ = run_one('RF | temporal | SMOTE | + post-accident feats', 'RF', True, True, extra=POST_ACCIDENT)
results.append(row)
df_res = pd.DataFrame(results).round(4)
df_res.to_csv('output/model_results_deployable.csv', index=False)
df_res[['run', 'n_features', 'accuracy', 'f1_macro', 'recall_macro', 'auc_roc_ovr', 'recall_sev4']]

In [ ]:
# Export: bundle, confusion matrix, generated Angular schema, sample input, version pins.
import json, sys, shutil, joblib, sklearn
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

f = fitted[KEEP]
pipe, features = f['pipe'], f['cols']

# 1. The pipeline must reproduce the evaluation predictions from RAW input rows.
check = pipe.predict(Xd_test[features].iloc[:2000])
assert (check == f['pred'][:2000]).all(), "Pipeline output differs from evaluation predictions"
print("Pipeline reproduces evaluation predictions on raw inputs: OK")

# 2. Bundle in the format the lab manual expects.
bundle = {"pipeline": pipe, "features": features, "task": "classification",
          "model_version": "1.0", "classes": [1, 2, 3, 4]}
joblib.dump(bundle, "output/model_bundle.joblib", compress=3)
size_mb = os.path.getsize("output/model_bundle.joblib") / 1e6
print(f"model_bundle.joblib: {size_mb:.1f} MB", "<-- TOO BIG for GitHub, raise min_samples_leaf" if size_mb > 90 else "(OK, under 100 MB)")

# 3. Confusion matrix (Issue #2).
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(yd_test, f['pred'], labels=[1, 2, 3, 4], ax=ax, colorbar=False)
ax.set_title(KEEP)
fig.savefig("output/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# 4. Angular feature-schema.ts generated from the real data (manual Stage 9.1).
pre = pipe.named_steps['preprocess']
cat_cols = pre.transformers_[0][2]
cat_options = dict(zip(cat_cols, [list(map(str, c)) for c in pre.named_transformers_['cat'].categories_]))
entries = []
for c in features:
    label = c.replace('_', ' ')
    if c in cat_options:
        e = {"name": c, "label": label, "kind": "category", "required": True, "options": cat_options[c]}
    elif c in BINARY:                                  # 0/1 flags are sent as numbers
        e = {"name": c, "label": label + " (0 or 1)", "kind": "number", "required": True, "min": 0, "max": 1}
    else:
        e = {"name": c, "label": label, "kind": "number", "required": True,
             "min": float(np.floor(Xd_train[c].min() * 100) / 100),
             "max": float(np.ceil(Xd_train[c].max() * 100) / 100)}
    entries.append(e)

ts = """export type Feature = {
  name: string;
  label: string;
  kind: 'number' | 'category';
  required: boolean;
  options?: string[];
  min?: number;
  max?: number;
};

// Generated from the trained model: %d raw input fields, in model order.
export const FEATURES: Feature[] = [
%s
];
""" % (len(entries), ",\n".join("  " + json.dumps(e) for e in entries))
open("output/feature-schema.ts", "w", encoding="utf-8").write(ts)

# 5. One non-sensitive sample row (manual Stage 5.1) and version pins for requirements.txt.
Xd_test[features].head(1).to_csv("output/sample_inputs.csv", index=False)
pins = [f"scikit-learn=={sklearn.__version__}", f"pandas=={pd.__version__}",
        f"numpy=={np.__version__}", f"joblib=={joblib.__version__}"]
open("output/version_pins.txt", "w").write("python " + sys.version.split()[0] + "\n" + "\n".join(pins) + "\n")
print("\n".join(pins), "| python", sys.version.split()[0])

# 6. Download everything in one zip.
shutil.make_archive("deploy_artifacts", "zip", "output")
try:
    from google.colab import files
    files.download("deploy_artifacts.zip")
except ImportError:
    print("Not in Colab: find deploy_artifacts.zip in the working folder")